# Stage 1 Modeling V3

这版 notebook 在 `v2` 的基础上继续向正式链路推进，重点补上：
- `raw` vs `calibrated` probability 对比
- formal `OOF P_taco` 输出
- formal `primary hold-out P_taco` 输出
- 为后续 `Stage 2` 聚合准备 event-level 结果表

这一版的主要目标不是再做新的特征工程，而是把 `Stage 1` 的概率产出做完整。

## Scope

本 notebook 采用以下实现策略：
- `outer / inner / calibration` 全部以 `feature_anchor_date` 为 trading-day 切分锚点
- 所有 CV split 统一采用 `segmented / gap-aware + purge / embargo`
- 主链正式 `primary holdout = 2025-07-15 ~ 2025-10-25`
- 候选模型保持不变：`Logistic Regression`、`XGBoost Classifier`、`Linear SVM`
- `Logistic` 与 `XGBoost` 同时输出 `raw`、`sigmoid`、`isotonic`
- `Linear SVM` 输出 `sigmoid`、`isotonic`
- 导出 all-variant 和 selected-variant 两套 OOF / hold-out 概率

说明：
- 当前 train pool 含 `first_term + second_term` 两段，因此 validation fold 不允许跨过中间大 gap
- `selected variant` 只在 calibrated variants 中比较，并以 `PR-AUC` 为第一优先；`Brier / ECE` 只作辅助判断与 tie-break
- `smoke` 档位用于先检查流程；若结果稳定，再切到 `protocol`

## Runtime Knob

这版 notebook 提供两个档位：
- `SEARCH_PROFILE = "smoke"`：默认，先快速确认流程和导出结果
- `SEARCH_PROFILE = "protocol"`：按 protocol 粗网格跑第一轮正式版本，不做大规模扩展搜索

# Main Rerun Stage 1 Runtime Config

这个 notebook 固定用于本次 `main rerun` 的 Stage 1：

- `SPLIT_SCHEME = "primary_holdout"`
- 正式 `primary holdout = 2025-07-15 ~ 2025-10-25`
- `STAGE1_OUTER_N_SPLITS = 5`
- 默认 `STAGE1_SEARCH_PROFILE = "smoke"`
- 输出目录固定到 `main_rerun/artifacts/stage1/`
- 不覆盖旧主链 `stage1/artifacts/v3/`

In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "plan_v2.md").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)

MAIN_RERUN_STAGE1_DIR = PROJECT_ROOT / "main_rerun" / "artifacts" / "stage1"
MAIN_RERUN_STAGE1_DIR.mkdir(parents=True, exist_ok=True)

os.environ["STAGE_SPLIT_SCHEME"] = "primary_holdout"
os.environ["STAGE1_OUTER_N_SPLITS"] = "5"
os.environ.setdefault("STAGE1_SEARCH_PROFILE", "smoke")
os.environ["STAGE1_RESULTS_DIR"] = str(MAIN_RERUN_STAGE1_DIR)
os.environ["STAGE1_HOLDOUT_ARTIFACT_STEM"] = "primary_holdout"
os.environ["STAGE1_HOLDOUT_SUMMARY_LABEL"] = "primary_holdout"

print("Project root:", PROJECT_ROOT)
print("Main rerun Stage 1 output dir:", MAIN_RERUN_STAGE1_DIR)
print("STAGE_SPLIT_SCHEME =", os.environ["STAGE_SPLIT_SCHEME"])
print("STAGE1_OUTER_N_SPLITS =", os.environ["STAGE1_OUTER_N_SPLITS"])
print("STAGE1_SEARCH_PROFILE =", os.environ["STAGE1_SEARCH_PROFILE"])

In [ ]:
from pathlib import Path
import json
import os

os.environ["MPLCONFIGDIR"] = str(Path.cwd() / ".mpl-cache")

import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="talk")

DATA_PATH = Path("data/integration/event_stage1_aligned_features_v1.csv")
RESULTS_DIR = Path(os.environ.get("STAGE1_RESULTS_DIR", "stage1/artifacts/v3"))
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
HOLDOUT_ARTIFACT_STEM = os.environ.get("STAGE1_HOLDOUT_ARTIFACT_STEM", "primary_holdout")
HOLDOUT_SUMMARY_LABEL = os.environ.get("STAGE1_HOLDOUT_SUMMARY_LABEL", HOLDOUT_ARTIFACT_STEM)
HOLDOUT_DISPLAY_TITLE = HOLDOUT_SUMMARY_LABEL.replace("_", " ").title()

RANDOM_STATE = 42
SEARCH_PROFILE = os.environ.get("STAGE1_SEARCH_PROFILE", "smoke")

OUTER_N_SPLITS = int(os.environ.get("STAGE1_OUTER_N_SPLITS", "5"))
OUTER_PURGE_DAYS = 2
OUTER_EMBARGO_DAYS = 5

INNER_N_SPLITS = 3
INNER_PURGE_DAYS = 2
INNER_EMBARGO_DAYS = 2

CALIBRATION_N_SPLITS = 3
CALIBRATION_PURGE_DAYS = 2
CALIBRATION_EMBARGO_DAYS = 2

MIN_INITIAL_TRAIN_DAYS = 180
MIN_INITIAL_TRAIN_POSITIVES = 10
GAP_SPLIT_DAYS = 30
LATER_SEGMENT_MIN_PREFIX_DAYS = 30
MIN_VALID_DAYS_PER_FOLD = 20

TOP_K_FRACS = [0.01, 0.03, 0.05, 0.10]

In [ ]:
SPLIT_SCHEME = os.environ.get("STAGE_SPLIT_SCHEME", "primary_holdout")

TRAIN_LABEL = "train_pool"
HOLDOUT_LABEL = os.environ.get("STAGE1_HOLDOUT_SUMMARY_LABEL", "primary_holdout")
OUTSIDE_LABEL = "outside_main_sample"

SPLIT_SCHEMES = {
    "primary_holdout": {
        "train_ranges": [
            ("2017-01-20", "2021-01-08"),
            ("2025-01-20", "2025-07-14"),
        ],
        "holdout_range": ("2025-07-15", "2025-10-25"),
    },
    "first_term_final_year_holdout": {
        "train_ranges": [
            ("2017-01-20", "2019-06-30"),
        ],
        "holdout_range": ("2019-07-01", "2020-02-28"),
    },
}


def describe_split_scheme(split_scheme):
    scheme = SPLIT_SCHEMES[split_scheme]
    return {
        "split_scheme": split_scheme,
        "train_ranges": scheme["train_ranges"],
        "holdout_range": scheme["holdout_range"],
        "train_label": TRAIN_LABEL,
        "holdout_label": HOLDOUT_LABEL,
    }


def build_event_split_masks(event_time_values, split_scheme=SPLIT_SCHEME):
    scheme = SPLIT_SCHEMES[split_scheme]
    event_dates = pd.to_datetime(pd.Series(event_time_values), format="mixed", utc=True).dt.normalize()

    train_mask = pd.Series(False, index=event_dates.index)
    for start_text, end_text in scheme["train_ranges"]:
        start = pd.Timestamp(start_text, tz="UTC")
        end = pd.Timestamp(end_text, tz="UTC")
        train_mask |= (event_dates >= start) & (event_dates <= end)

    holdout_start, holdout_end = scheme["holdout_range"]
    holdout_mask = (
        (event_dates >= pd.Timestamp(holdout_start, tz="UTC"))
        & (event_dates <= pd.Timestamp(holdout_end, tz="UTC"))
    )
    return train_mask, holdout_mask


print("Split scheme:", describe_split_scheme(SPLIT_SCHEME))

In [ ]:
NUMERIC_FEATURES = [
    "finbert_pos",
    "finbert_neu",
    "finbert_neg",
    "candidate_priority_high",
    "keyword_score",
    "in_market_hours",
    "spx_ret_1d",
    "dxy_ret_1d",
    "vix_change_1d",
    "vix_change_5d",
    "real_yield_change_1d",
    "real_yield_change_5d",
    "spx_drawdown_1d",
    "spx_drawdown_5d",
    "vix_ma_5d",
    "gold_spx_corr_20d",
    "trend_tariff_z_60d",
    "trend_inflation_z_60d",
    "trend_war_z_60d",
    "trend_tariff_spike",
    "trend_inflation_spike",
    "trend_war_spike",
]

CATEGORICAL_FEATURES = [
    "term_id",
    "source",
]

LEAKAGE_PRONE_COLUMNS = [
    "follow_up_count_all_48h",
    "follow_up_count_relevant_48h",
    "context_mode",
    "has_evidence_span",
    "review_flag",
    "review_reason",
    "review_flag_yes",
    "rule_label",
    "rule_llm_conflict",
    "evidence_span",
    "rule_text",
    "source_text",
    "key_event_id",
    "audit_tier",
    "confidence",
]

MODEL_NAMES = [
    "logistic_regression",
    "xgboost_classifier",
    "linear_svm",
]

CALIBRATION_METHODS = ["sigmoid", "isotonic"]


def compute_ece(y_true, y_prob, n_bins=10):
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(y_prob, bins[1:-1], right=True)
    ece = 0.0
    for idx in range(n_bins):
        mask = bin_ids == idx
        if not mask.any():
            continue
        observed = y_true[mask].mean()
        predicted = y_prob[mask].mean()
        ece += np.abs(observed - predicted) * mask.mean()
    return float(ece)


def top_k_capture_table(y_true, y_prob, fracs):
    out = []
    total_pos = int(np.sum(y_true))
    ranked = pd.DataFrame({"y_true": y_true, "y_prob": y_prob}).sort_values(
        "y_prob", ascending=False
    )
    for frac in fracs:
        k = max(1, int(np.ceil(len(ranked) * frac)))
        top_slice = ranked.head(k)
        captured = int(top_slice["y_true"].sum())
        out.append(
            {
                "top_k_frac": frac,
                "k": k,
                "captured_positives": captured,
                "capture_rate": captured / total_pos if total_pos else np.nan,
                "precision_at_k": top_slice["y_true"].mean(),
            }
        )
    return pd.DataFrame(out)


def compute_binary_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    metrics = {
        "pr_auc": average_precision_score(y_true, y_prob),
        "brier": brier_score_loss(y_true, y_prob),
        "ece": compute_ece(y_true, y_prob),
        "positive_rate": float(np.mean(y_true)),
        "precision_at_0p5": precision_score(y_true, y_pred, zero_division=0),
        "recall_at_0p5": recall_score(y_true, y_pred, zero_division=0),
        "f1_at_0p5": f1_score(y_true, y_pred, zero_division=0),
    }
    if len(np.unique(y_true)) > 1:
        metrics["roc_auc"] = roc_auc_score(y_true, y_prob)
    else:
        metrics["roc_auc"] = np.nan
    return metrics


def make_preprocessor(numeric_cols, categorical_cols):
    return ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric_cols,
            ),
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore")),
                    ]
                ),
                categorical_cols,
            ),
        ],
        remainder="drop",
    )


def split_day_segments(unique_days, gap_days=30):
    unique_days = pd.Index(sorted(pd.Index(unique_days).unique()))
    if len(unique_days) == 0:
        return []

    segments = []
    start = 0
    diffs = np.diff(unique_days.view("int64")) / (24 * 3600 * 1e9)
    for idx, gap in enumerate(diffs, start=1):
        if gap > gap_days:
            segments.append(unique_days[start:idx])
            start = idx
    segments.append(unique_days[start:])
    return segments


def allocate_splits_to_segments(
    segments,
    day_positive_by_segment,
    n_splits,
    min_initial_train_days,
    purge_days,
    embargo_days,
    min_valid_days,
    min_prefix_days_later_segment,
):
    eligible = []
    for seg_idx, segment in enumerate(segments):
        required_prefix_days = (
            min_initial_train_days if seg_idx == 0 else min_prefix_days_later_segment
        )
        required_days = required_prefix_days + purge_days + min_valid_days
        if len(segment) < required_days:
            continue

        positive_days = int((day_positive_by_segment[seg_idx] > 0).sum())
        if positive_days == 0:
            continue

        capacity = int(
            (len(segment) - required_days) // max(min_valid_days + embargo_days, 1)
        ) + 1
        capacity = min(capacity, positive_days)
        if capacity <= 0:
            continue

        eligible.append(
            {
                "segment_idx": seg_idx,
                "segment_len": len(segment),
                "positive_days": positive_days,
                "capacity": capacity,
            }
        )

    if not eligible:
        raise ValueError("No eligible day segments available for the requested split design.")

    allocation = {item["segment_idx"]: 0 for item in eligible}
    remaining = n_splits

    for item in eligible:
        if remaining <= 0:
            break
        allocation[item["segment_idx"]] += 1
        remaining -= 1

    while remaining > 0:
        best_choice = None
        for item in eligible:
            current = allocation[item["segment_idx"]]
            if current >= item["capacity"]:
                continue

            score = (item["positive_days"] / (current + 1), item["segment_len"])
            if best_choice is None or score > best_choice[0]:
                best_choice = (score, item["segment_idx"])

        if best_choice is None:
            break

        allocation[best_choice[1]] += 1
        remaining -= 1

    if remaining > 0:
        raise ValueError("Segment capacities cannot satisfy the requested number of splits.")

    return allocation


def build_segmented_purged_splits(
    frame,
    date_col,
    label_col,
    n_splits,
    purge_days,
    embargo_days,
    min_initial_train_days,
    min_initial_train_positives,
    gap_days,
    min_prefix_days_later_segment,
    min_valid_days,
):
    work = frame.copy()
    sort_cols = [col for col in ["event_time_utc", "event_id"] if col in work.columns]
    work = work.sort_values(sort_cols).reset_index(drop=True)
    work["_cv_day"] = work[date_col].dt.normalize()

    unique_days = pd.Index(sorted(work["_cv_day"].dropna().unique()))
    segments = split_day_segments(unique_days, gap_days=gap_days)
    if len(segments) == 0:
        raise ValueError("No trading days available to build segmented splits.")

    day_positive_series = work.groupby("_cv_day")[label_col].sum()
    day_positive_by_segment = [
        day_positive_series.reindex(segment, fill_value=0).to_numpy()
        for segment in segments
    ]

    allocation = allocate_splits_to_segments(
        segments=segments,
        day_positive_by_segment=day_positive_by_segment,
        n_splits=n_splits,
        min_initial_train_days=min_initial_train_days,
        purge_days=purge_days,
        embargo_days=embargo_days,
        min_valid_days=min_valid_days,
        min_prefix_days_later_segment=min_prefix_days_later_segment,
    )

    splits = []
    prior_days = []
    prior_positive_total = 0
    fold_id = 1

    for seg_idx, segment in enumerate(segments):
        n_segment_splits = allocation.get(seg_idx, 0)
        day_positive = day_positive_by_segment[seg_idx]

        if n_segment_splits == 0:
            prior_days.extend(list(segment))
            prior_positive_total += int(day_positive.sum())
            continue

        prefix_days_required = (
            min_initial_train_days if seg_idx == 0 else min_prefix_days_later_segment
        )
        remaining_positive_needed = max(min_initial_train_positives - prior_positive_total, 0)
        if remaining_positive_needed > 0:
            cumulative_positive = np.cumsum(day_positive)
            positive_hit = np.where(cumulative_positive >= remaining_positive_needed)[0]
            if len(positive_hit) == 0:
                raise ValueError(
                    f"Segment {seg_idx} cannot satisfy the initial positive-count requirement."
                )
            prefix_days_required = max(prefix_days_required, int(positive_hit[0]) + 1)

        validation_start = prefix_days_required + purge_days
        positive_positions = np.flatnonzero(day_positive > 0)
        positive_positions = positive_positions[positive_positions >= validation_start]
        if len(positive_positions) < n_segment_splits:
            raise ValueError(
                f"Segment {seg_idx} does not have enough positive trading days for {n_segment_splits} folds."
            )

        min_validation_days = max(
            min_valid_days,
            int(np.ceil((len(segment) - validation_start) / max(n_segment_splits * 3, 1))),
        )

        for local_fold_id in range(1, n_segment_splits + 1):
            remaining_folds_after = n_segment_splits - local_fold_id
            remaining_positive_positions = positive_positions[positive_positions >= validation_start]
            if len(remaining_positive_positions) < (remaining_folds_after + 1):
                raise ValueError(
                    f"Segment {seg_idx} cannot keep one positive day for each remaining fold."
                )

            target_positive_count = int(
                np.ceil(len(remaining_positive_positions) / (remaining_folds_after + 1))
            )
            target_positive_count = max(1, target_positive_count)

            val_start_pos = validation_start
            candidate_end_pos = remaining_positive_positions[target_positive_count - 1]
            candidate_end_pos = max(candidate_end_pos, val_start_pos + min_validation_days - 1)

            if remaining_folds_after > 0:
                reserve_first_pos = remaining_positive_positions[-remaining_folds_after]
                max_allowed_end = reserve_first_pos - embargo_days - 1
                candidate_end_pos = min(candidate_end_pos, max_allowed_end)
            else:
                candidate_end_pos = max(candidate_end_pos, remaining_positive_positions[-1])

            val_end_pos = min(candidate_end_pos, len(segment) - 1)
            if val_end_pos < val_start_pos:
                raise ValueError(f"Segment {seg_idx} produced an invalid validation window.")

            train_end_pos = val_start_pos - purge_days - 1
            train_days = pd.Index(list(prior_days) + list(segment[: train_end_pos + 1]))
            valid_days = segment[val_start_pos : val_end_pos + 1]

            train_idx = np.flatnonzero(work["_cv_day"].isin(train_days))
            valid_idx = np.flatnonzero(work["_cv_day"].isin(valid_days))

            train_positive = int(work.loc[train_idx, label_col].sum())
            valid_positive = int(work.loc[valid_idx, label_col].sum())
            if train_positive < min_initial_train_positives:
                raise ValueError(
                    f"Fold {fold_id} training split cannot satisfy the minimum positive count."
                )

            purge_start = segment[train_end_pos + 1] if train_end_pos + 1 < len(segment) else pd.NaT
            purge_end = segment[val_start_pos - 1] if val_start_pos - 1 < len(segment) else pd.NaT
            embargo_start = segment[val_end_pos + 1] if val_end_pos + 1 < len(segment) else pd.NaT
            embargo_end_pos = min(val_end_pos + embargo_days, len(segment) - 1)
            embargo_end = segment[embargo_end_pos] if val_end_pos + 1 < len(segment) else pd.NaT

            splits.append(
                {
                    "fold_id": fold_id,
                    "segment_idx": seg_idx,
                    "train_idx": train_idx,
                    "valid_idx": valid_idx,
                    "segment_start_day": segment.min(),
                    "segment_end_day": segment.max(),
                    "train_start_day": train_days.min(),
                    "train_end_day": train_days.max(),
                    "valid_start_day": valid_days.min(),
                    "valid_end_day": valid_days.max(),
                    "purge_start_day": purge_start,
                    "purge_end_day": purge_end,
                    "embargo_start_day": embargo_start,
                    "embargo_end_day": embargo_end,
                    "train_day_count": len(train_days),
                    "valid_day_count": len(valid_days),
                    "train_rows": len(train_idx),
                    "valid_rows": len(valid_idx),
                    "train_positive": train_positive,
                    "valid_positive": valid_positive,
                }
            )

            fold_id += 1
            validation_start = val_end_pos + 1 + embargo_days

        prior_days.extend(list(segment))
        prior_positive_total += int(day_positive.sum())

    if len(splits) != n_splits:
        raise ValueError(
            f"Expected {n_splits} segmented folds, but built {len(splits)} folds."
        )

    return work.drop(columns="_cv_day"), splits


def build_base_bundle(model_name, numeric_cols, categorical_cols, search_profile, scale_pos_weight):
    preprocessor = make_preprocessor(numeric_cols, categorical_cols)

    if search_profile == "protocol":
        grids = {
            "logistic_regression": {
                "model__penalty": ["l1", "l2"],
                "model__C": [0.01, 0.1, 1],
            },
            "xgboost_classifier": {
                "model__n_estimators": [200, 400],
                "model__max_depth": [2, 3],
                "model__learning_rate": [0.03, 0.08],
                "model__min_child_weight": [1, 5],
                "model__subsample": [0.8],
                "model__colsample_bytree": [0.8],
                "model__reg_lambda": [1],
                "model__scale_pos_weight": [scale_pos_weight],
            },
            "linear_svm": {
                "model__C": [0.1, 1, 10],
                "model__loss": ["squared_hinge"],
            },
        }
    else:
        grids = {
            "logistic_regression": {
                "model__penalty": ["l1", "l2"],
                "model__C": [0.01, 0.1],
            },
            "xgboost_classifier": {
                "model__n_estimators": [200],
                "model__max_depth": [2, 3],
                "model__learning_rate": [0.05],
                "model__min_child_weight": [1, 5],
                "model__subsample": [0.8],
                "model__colsample_bytree": [0.8],
                "model__reg_lambda": [1],
                "model__scale_pos_weight": [scale_pos_weight],
            },
            "linear_svm": {
                "model__C": [0.1, 1],
                "model__loss": ["squared_hinge"],
            },
        }

    if model_name == "logistic_regression":
        estimator = LogisticRegression(
            solver="liblinear",
            class_weight="balanced",
            max_iter=2000,
            random_state=RANDOM_STATE,
        )
    elif model_name == "xgboost_classifier":
        estimator = XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=1,
            tree_method="hist",
            verbosity=0,
        )
    elif model_name == "linear_svm":
        estimator = LinearSVC(
            class_weight="balanced",
            max_iter=5000,
            random_state=RANDOM_STATE,
        )
    else:
        raise ValueError(f"Unknown model_name: {model_name}")

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", estimator),
        ]
    )
    return pipeline, grids[model_name]


def choose_cv_for_subset(
    X_train,
    y_train,
    n_splits,
    purge_days,
    embargo_days,
    min_initial_train_days,
    min_initial_train_positives,
):
    inner_frame = X_train.copy()
    inner_frame["label_retreat"] = y_train
    inner_frame["event_time_utc"] = pd.to_datetime(inner_frame["event_time_utc"], utc=True)
    inner_frame["feature_anchor_date"] = pd.to_datetime(inner_frame["feature_anchor_date"], utc=True)

    try:
        _, inner_splits = build_segmented_purged_splits(
            frame=inner_frame,
            date_col="feature_anchor_date",
            label_col="label_retreat",
            n_splits=n_splits,
            purge_days=purge_days,
            embargo_days=embargo_days,
            min_initial_train_days=min_initial_train_days,
            min_initial_train_positives=min_initial_train_positives,
            gap_days=GAP_SPLIT_DAYS,
            min_prefix_days_later_segment=min(
                LATER_SEGMENT_MIN_PREFIX_DAYS,
                max(min_initial_train_days // 2, 10),
            ),
            min_valid_days=max(10, min(MIN_VALID_DAYS_PER_FOLD, max(n_splits * 5, 10))),
        )
        cv = [(item["train_idx"], item["valid_idx"]) for item in inner_splits]
        strategy = "segmented_purged"
    except ValueError:
        cv = TimeSeriesSplit(n_splits=n_splits)
        strategy = "time_series_fallback"
    return cv, strategy


def run_grid_search(X_train, y_train, model_name, numeric_cols, categorical_cols, search_profile):
    positive_count = int(np.sum(y_train))
    negative_count = int(len(y_train) - positive_count)
    if positive_count == 0:
        raise ValueError("Training split has no positive samples.")

    scale_pos_weight = negative_count / positive_count
    estimator, param_grid = build_base_bundle(
        model_name=model_name,
        numeric_cols=numeric_cols,
        categorical_cols=categorical_cols,
        search_profile=search_profile,
        scale_pos_weight=scale_pos_weight,
    )

    cv, cv_strategy = choose_cv_for_subset(
        X_train=X_train,
        y_train=y_train,
        n_splits=INNER_N_SPLITS,
        purge_days=INNER_PURGE_DAYS,
        embargo_days=INNER_EMBARGO_DAYS,
        min_initial_train_days=max(90, int(MIN_INITIAL_TRAIN_DAYS * 0.5)),
        min_initial_train_positives=max(4, int(MIN_INITIAL_TRAIN_POSITIVES * 0.4)),
    )

    search = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        scoring="average_precision",
        cv=cv,
        refit=True,
        n_jobs=1,
        verbose=0,
    )
    search.fit(X_train, y_train)
    return search, cv_strategy


def get_probability_variants(
    search,
    model_name,
    X_train,
    y_train,
    X_scored,
):
    variants = {}
    meta = []

    best_base = clone(search.best_estimator_)
    best_base.fit(X_train, y_train)

    if model_name in {"logistic_regression", "xgboost_classifier"}:
        variants["raw"] = best_base.predict_proba(X_scored)[:, 1]
        meta.append({"prob_version": "raw", "calibration_cv_strategy": "not_applicable"})

    calibration_cv, calibration_strategy = choose_cv_for_subset(
        X_train=X_train,
        y_train=y_train,
        n_splits=CALIBRATION_N_SPLITS,
        purge_days=CALIBRATION_PURGE_DAYS,
        embargo_days=CALIBRATION_EMBARGO_DAYS,
        min_initial_train_days=max(90, int(MIN_INITIAL_TRAIN_DAYS * 0.5)),
        min_initial_train_positives=max(4, int(MIN_INITIAL_TRAIN_POSITIVES * 0.4)),
    )

    for method in CALIBRATION_METHODS:
        calibrated = CalibratedClassifierCV(
            estimator=clone(search.best_estimator_),
            method=method,
            cv=calibration_cv,
        )
        calibrated.fit(X_train, y_train)
        variants[method] = calibrated.predict_proba(X_scored)[:, 1]
        meta.append(
            {
                "prob_version": method,
                "calibration_cv_strategy": calibration_strategy,
            }
        )

    return variants, meta


def save_progress_csvs(mapping):
    for path, frame in mapping.items():
        if frame is None:
            continue
        if hasattr(frame, "empty") and frame.empty:
            continue
        frame.to_csv(path, index=False)

In [ ]:
df = pd.read_csv(DATA_PATH)
df["event_time_utc"] = pd.to_datetime(df["event_time_utc"], format="mixed", utc=True)
df["event_time_ny"] = pd.to_datetime(df["event_time_ny"], format="mixed", utc=True, errors="coerce")
df["feature_anchor_date"] = pd.to_datetime(df["feature_anchor_date"], format="mixed", utc=True, errors="coerce")
df["target_trade_date"] = pd.to_datetime(df["target_trade_date"], format="mixed", utc=True, errors="coerce")

df = df.sort_values(["event_time_utc", "event_id"]).reset_index(drop=True)

print(f"Loaded {len(df):,} rows and {df.shape[1]} columns from {DATA_PATH}")
print("Search profile:", SEARCH_PROFILE)

In [ ]:
train_mask, holdout_mask = build_event_split_masks(
    df["event_time_utc"],
    split_scheme=SPLIT_SCHEME,
)

active_numeric = [col for col in NUMERIC_FEATURES if col in df.columns]
active_categorical = [col for col in CATEGORICAL_FEATURES if col in df.columns]
active_features = active_numeric + active_categorical

constant_cols = [
    col for col in active_features if df.loc[train_mask, col].nunique(dropna=False) <= 1
]
active_numeric = [col for col in active_numeric if col not in constant_cols]
active_categorical = [col for col in active_categorical if col not in constant_cols]
active_features = active_numeric + active_categorical

modeling_cols = [
    "event_id",
    "text_id",
    "event_time_utc",
    "feature_anchor_date",
    "target_trade_date",
    "term_id",
    "source",
    "label_retreat",
] + active_features
modeling_cols = list(dict.fromkeys(modeling_cols))

train_df = df.loc[train_mask, modeling_cols].copy().sort_values(["event_time_utc", "event_id"]).reset_index(drop=True)
holdout_df = df.loc[holdout_mask, modeling_cols].copy().sort_values(["event_time_utc", "event_id"]).reset_index(drop=True)

split_summary = pd.DataFrame(
    [
        {
            "split": "train_pool",
            "n_rows": len(train_df),
            "n_positive": int(train_df["label_retreat"].sum()),
            "positive_rate": train_df["label_retreat"].mean(),
            "start": train_df["event_time_utc"].min(),
            "end": train_df["event_time_utc"].max(),
        },
        {
            "split": HOLDOUT_SUMMARY_LABEL,
            "n_rows": len(holdout_df),
            "n_positive": int(holdout_df["label_retreat"].sum()),
            "positive_rate": holdout_df["label_retreat"].mean(),
            "start": holdout_df["event_time_utc"].min(),
            "end": holdout_df["event_time_utc"].max(),
        },
    ]
)
display(split_summary)
print("Active features:", active_features)
print("Excluded leakage-prone columns:", ", ".join([c for c in LEAKAGE_PRONE_COLUMNS if c in df.columns]))

In [ ]:
train_df_cv, outer_splits = build_segmented_purged_splits(
    frame=train_df,
    date_col="feature_anchor_date",
    label_col="label_retreat",
    n_splits=OUTER_N_SPLITS,
    purge_days=OUTER_PURGE_DAYS,
    embargo_days=OUTER_EMBARGO_DAYS,
    min_initial_train_days=MIN_INITIAL_TRAIN_DAYS,
    min_initial_train_positives=MIN_INITIAL_TRAIN_POSITIVES,
    gap_days=GAP_SPLIT_DAYS,
    min_prefix_days_later_segment=LATER_SEGMENT_MIN_PREFIX_DAYS,
    min_valid_days=MIN_VALID_DAYS_PER_FOLD,
)

outer_split_df = pd.DataFrame(outer_splits).drop(columns=["train_idx", "valid_idx"])
display(outer_split_df)

In [ ]:
outer_metric_rows = []
outer_prediction_rows = []
search_rows = []

for fold in outer_splits:
    fold_id = fold["fold_id"]
    outer_train = train_df_cv.iloc[fold["train_idx"]].copy()
    outer_valid = train_df_cv.iloc[fold["valid_idx"]].copy()

    y_outer_train = outer_train["label_retreat"].astype(int).to_numpy()
    y_outer_valid = outer_valid["label_retreat"].astype(int).to_numpy()

    feature_train = outer_train[["event_time_utc", "feature_anchor_date"] + active_features].copy()
    feature_valid = outer_valid[["event_time_utc", "feature_anchor_date"] + active_features].copy()

    print(
        f"Fold {fold_id}: train_rows={len(outer_train)}, train_pos={int(y_outer_train.sum())}, "
        f"valid_rows={len(outer_valid)}, valid_pos={int(y_outer_valid.sum())}"
    )

    for model_name in MODEL_NAMES:
        print(f"  searching {model_name} ...")
        search, inner_cv_strategy = run_grid_search(
            X_train=feature_train,
            y_train=y_outer_train,
            model_name=model_name,
            numeric_cols=active_numeric,
            categorical_cols=active_categorical,
            search_profile=SEARCH_PROFILE,
        )
        print(
            f"  best {model_name}: inner_pr_auc={float(search.best_score_):.6f}, "
            f"params={json.dumps(search.best_params_, sort_keys=True)}"
        )

        variants, variant_meta = get_probability_variants(
            search=search,
            model_name=model_name,
            X_train=feature_train,
            y_train=y_outer_train,
            X_scored=feature_valid,
        )

        search_rows.append(
            {
                "fold_id": fold_id,
                "model_name": model_name,
                "best_params": json.dumps(search.best_params_, sort_keys=True),
                "inner_best_pr_auc": float(search.best_score_),
                "inner_cv_strategy": inner_cv_strategy,
            }
        )

        meta_lookup = {row["prob_version"]: row for row in variant_meta}
        for prob_version, prob in variants.items():
            metrics = compute_binary_metrics(y_outer_valid, prob, threshold=0.5)
            outer_metric_rows.append(
                {
                    "fold_id": fold_id,
                    "model_name": model_name,
                    "prob_version": prob_version,
                    "inner_best_pr_auc": float(search.best_score_),
                    "inner_cv_strategy": inner_cv_strategy,
                    "calibration_cv_strategy": meta_lookup[prob_version]["calibration_cv_strategy"],
                    "train_rows": len(outer_train),
                    "train_positive": int(y_outer_train.sum()),
                    "valid_rows": len(outer_valid),
                    "valid_positive": int(y_outer_valid.sum()),
                    **metrics,
                }
            )

            pred_df = outer_valid[
                ["event_id", "text_id", "event_time_utc", "feature_anchor_date", "target_trade_date", "label_retreat"]
            ].copy()
            pred_df["fold_id"] = fold_id
            pred_df["model_name"] = model_name
            pred_df["prob_version"] = prob_version
            pred_df["p_retreat"] = prob
            pred_df["is_oof"] = 1
            outer_prediction_rows.append(pred_df)

        save_progress_csvs(
            {
                RESULTS_DIR / "_progress_outer_fold_metrics.csv": pd.DataFrame(outer_metric_rows),
                RESULTS_DIR / "_progress_oof_predictions_all_variants.csv": pd.concat(outer_prediction_rows, ignore_index=True),
                RESULTS_DIR / "_progress_best_params_by_fold.csv": pd.DataFrame(search_rows),
            }
        )

outer_metrics_df = pd.DataFrame(outer_metric_rows)
outer_predictions_df = pd.concat(outer_prediction_rows, ignore_index=True)
search_df = pd.DataFrame(search_rows)

outer_summary_df = (
    outer_metrics_df.groupby(["model_name", "prob_version"])
    .agg(
        mean_pr_auc=("pr_auc", "mean"),
        std_pr_auc=("pr_auc", "std"),
        mean_brier=("brier", "mean"),
        mean_ece=("ece", "mean"),
        mean_roc_auc=("roc_auc", "mean"),
        mean_recall_0p5=("recall_at_0p5", "mean"),
        mean_precision_0p5=("precision_at_0p5", "mean"),
    )
    .reset_index()
    .sort_values(["mean_brier", "mean_ece", "mean_pr_auc"], ascending=[True, True, False])
)

display(outer_metrics_df)
display(outer_summary_df)

In [ ]:
calibrated_selection_pool = outer_summary_df[
    outer_summary_df["prob_version"].isin(CALIBRATION_METHODS)
].copy()
if calibrated_selection_pool.empty:
    raise ValueError("No calibrated variants available for Stage 1 final selection.")

selected_variant_df = (
    calibrated_selection_pool.sort_values(
        ["mean_pr_auc", "mean_brier", "mean_ece"],
        ascending=[False, True, True],
    )
    .head(1)
    .copy()
)
selected_variant_df["variant_key"] = (
    selected_variant_df["model_name"] + "__" + selected_variant_df["prob_version"]
)
selected_variant_df["selection_rule"] = (
    "calibrated_only__pr_auc_first__brier_ece_tiebreak"
)

display(selected_variant_df)
print("Final Stage 1 variant:", selected_variant_df.iloc[0]["variant_key"])

selected_oof_df = outer_predictions_df.merge(
    selected_variant_df[["model_name", "prob_version"]],
    how="inner",
    on=["model_name", "prob_version"],
)

selected_oof_df = selected_oof_df.sort_values(["event_time_utc", "event_id"]).reset_index(drop=True)
display(selected_oof_df.head())

In [ ]:
plt.figure(figsize=(12, 6))
plot_df = outer_summary_df.copy()
plot_df["model_variant"] = plot_df["model_name"] + " | " + plot_df["prob_version"]
sns.barplot(data=plot_df, x="model_variant", y="mean_pr_auc", palette="deep")
plt.title("Outer CV Mean PR-AUC by Model Variant")
plt.xlabel("model_variant")
plt.ylabel("mean PR-AUC")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
holdout_metric_rows = []
holdout_prediction_frames = []
holdout_topk_frames = []

feature_train_full = train_df_cv[["event_time_utc", "feature_anchor_date"] + active_features].copy()
y_train_full = train_df_cv["label_retreat"].astype(int).to_numpy()
feature_holdout = holdout_df[["event_time_utc", "feature_anchor_date"] + active_features].copy()
y_holdout = holdout_df["label_retreat"].astype(int).to_numpy()

for model_name in MODEL_NAMES:
    print(f"{HOLDOUT_SUMMARY_LABEL} refit for {model_name} ...")
    search, inner_cv_strategy = run_grid_search(
        X_train=feature_train_full,
        y_train=y_train_full,
        model_name=model_name,
        numeric_cols=active_numeric,
        categorical_cols=active_categorical,
        search_profile=SEARCH_PROFILE,
    )

    variants, variant_meta = get_probability_variants(
        search=search,
        model_name=model_name,
        X_train=feature_train_full,
        y_train=y_train_full,
        X_scored=feature_holdout,
    )
    meta_lookup = {row["prob_version"]: row for row in variant_meta}

    for prob_version, prob in variants.items():
        metrics = compute_binary_metrics(y_holdout, prob, threshold=0.5)
        holdout_metric_rows.append(
            {
                "model_name": model_name,
                "prob_version": prob_version,
                "best_params": json.dumps(search.best_params_, sort_keys=True),
                "cv_best_pr_auc": float(search.best_score_),
                "inner_cv_strategy": inner_cv_strategy,
                "calibration_cv_strategy": meta_lookup[prob_version]["calibration_cv_strategy"],
                **metrics,
            }
        )

        topk = top_k_capture_table(y_holdout, prob, TOP_K_FRACS)
        topk["model_name"] = model_name
        topk["prob_version"] = prob_version
        holdout_topk_frames.append(topk)

        pred_df = holdout_df[
            ["event_id", "text_id", "event_time_utc", "feature_anchor_date", "target_trade_date", "label_retreat"]
        ].copy()
        pred_df["model_name"] = model_name
        pred_df["prob_version"] = prob_version
        pred_df["p_retreat"] = prob
        pred_df["is_holdout"] = 1
        holdout_prediction_frames.append(pred_df)

    save_progress_csvs(
        {
            RESULTS_DIR / f"_progress_{HOLDOUT_ARTIFACT_STEM}_metrics_all_variants.csv": pd.DataFrame(holdout_metric_rows),
            RESULTS_DIR / f"_progress_{HOLDOUT_ARTIFACT_STEM}_topk_all_variants.csv": pd.concat(holdout_topk_frames, ignore_index=True),
            RESULTS_DIR / f"_progress_{HOLDOUT_ARTIFACT_STEM}_predictions_all_variants.csv": pd.concat(holdout_prediction_frames, ignore_index=True),
        }
    )

holdout_metrics_df = pd.DataFrame(holdout_metric_rows).sort_values(
    ["brier", "ece", "pr_auc"], ascending=[True, True, False]
)
holdout_topk_df = pd.concat(holdout_topk_frames, ignore_index=True)
holdout_predictions_df = pd.concat(holdout_prediction_frames, ignore_index=True)

selected_holdout_df = holdout_predictions_df.merge(
    selected_variant_df[["model_name", "prob_version"]],
    how="inner",
    on=["model_name", "prob_version"],
)

selected_holdout_metrics_df = holdout_metrics_df.merge(
    selected_variant_df[["model_name", "prob_version"]],
    how="inner",
    on=["model_name", "prob_version"],
)

display(holdout_metrics_df)
display(selected_holdout_metrics_df)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_df = selected_holdout_metrics_df.copy()
plot_df["model_variant"] = plot_df["model_name"] + " | " + plot_df["prob_version"]

sns.barplot(data=plot_df, x="model_variant", y="pr_auc", ax=axes[0], palette="muted")
axes[0].set_title(f"Selected {HOLDOUT_DISPLAY_TITLE} PR-AUC")
axes[0].set_xlabel("model_variant")
axes[0].set_ylabel("PR-AUC")
axes[0].tick_params(axis="x", rotation=20)

sns.barplot(data=plot_df, x="model_variant", y="brier", ax=axes[1], palette="crest")
axes[1].set_title(f"Selected {HOLDOUT_DISPLAY_TITLE} Brier")
axes[1].set_xlabel("model_variant")
axes[1].set_ylabel("Brier")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

In [ ]:
calibration_selection_path = RESULTS_DIR / "stage1_v3_selected_variants.csv"
outer_split_path = RESULTS_DIR / "stage1_v3_outer_splits.csv"
outer_metrics_path = RESULTS_DIR / "stage1_v3_outer_fold_metrics.csv"
outer_summary_path = RESULTS_DIR / "stage1_v3_outer_summary_metrics.csv"
outer_predictions_path = RESULTS_DIR / "stage1_v3_oof_predictions_all_variants.csv"
selected_oof_path = RESULTS_DIR / "stage1_v3_oof_predictions_selected.csv"
search_path = RESULTS_DIR / "stage1_v3_best_params_by_fold.csv"
holdout_metrics_path = RESULTS_DIR / f"stage1_v3_{HOLDOUT_ARTIFACT_STEM}_metrics_all_variants.csv"
selected_holdout_metrics_path = RESULTS_DIR / f"stage1_v3_{HOLDOUT_ARTIFACT_STEM}_metrics_selected.csv"
holdout_topk_path = RESULTS_DIR / f"stage1_v3_{HOLDOUT_ARTIFACT_STEM}_topk_all_variants.csv"
holdout_predictions_path = RESULTS_DIR / f"stage1_v3_{HOLDOUT_ARTIFACT_STEM}_predictions_all_variants.csv"
selected_holdout_predictions_path = RESULTS_DIR / f"stage1_v3_{HOLDOUT_ARTIFACT_STEM}_predictions_selected.csv"

selected_variant_df.to_csv(calibration_selection_path, index=False)
outer_split_df.to_csv(outer_split_path, index=False)
outer_metrics_df.to_csv(outer_metrics_path, index=False)
outer_summary_df.to_csv(outer_summary_path, index=False)
outer_predictions_df.to_csv(outer_predictions_path, index=False)
selected_oof_df.to_csv(selected_oof_path, index=False)
search_df.to_csv(search_path, index=False)
holdout_metrics_df.to_csv(holdout_metrics_path, index=False)
selected_holdout_metrics_df.to_csv(selected_holdout_metrics_path, index=False)
holdout_topk_df.to_csv(holdout_topk_path, index=False)
holdout_predictions_df.to_csv(holdout_predictions_path, index=False)
selected_holdout_df.to_csv(selected_holdout_predictions_path, index=False)

print("Saved:")
for path in [
    calibration_selection_path,
    outer_split_path,
    outer_metrics_path,
    outer_summary_path,
    outer_predictions_path,
    selected_oof_path,
    search_path,
    holdout_metrics_path,
    selected_holdout_metrics_path,
    holdout_topk_path,
    holdout_predictions_path,
    selected_holdout_predictions_path,
]:
    print(" -", path)

## What To Read First

如果你想快速判断这版 `Stage 1` 是否可以往 `Stage 2` 走，建议优先看：
1. `stage1_v3_selected_variants.csv`
2. `stage1_v3_{HOLDOUT_ARTIFACT_STEM}_metrics_selected.csv`
3. `stage1_v3_oof_predictions_selected.csv`
4. `stage1_v3_{HOLDOUT_ARTIFACT_STEM}_predictions_selected.csv`